<a href="https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/meharalirajar060-codeee/Flyrank_Internship_ML_MAR"
REPO_DIR = "Flyrank_Internship_ML_MAR"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())

Working dir: /content/Flyrank_Internship_ML_MAR


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""I'm choosing Lane 2: Refresh / Content Opportunity Scoring. In Weeks 1–2 I already ran the starter pipeline and found that a learned model (random forest) beat the hand-written baseline rule at Precision@50 (0.240 → 0.740, a ~3.1x lift) — client-holdout validated. I also found in my "your turn" work that tree depth beyond 2 barely improves held-out precision (0.56→0.60 across depth 2–4), suggesting the useful signal is shallow and interpretable. This lane lets me build directly on that evidence rather than start a new direction from zero."""


'I\'m choosing Lane 2: Refresh / Content Opportunity Scoring. In Weeks 1–2 I already ran the starter pipeline and found that a learned model (random forest) beat the hand-written baseline rule at Precision@50 (0.240 → 0.740, a ~3.1x lift) — client-holdout validated. I also found in my "your turn" work that tree depth beyond 2 barely improves held-out precision (0.56→0.60 across depth 2–4), suggesting the useful signal is shallow and interpretable. This lane lets me build directly on that evidence rather than start a new direction from zero.'

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Question: Which pages should a content reviewer look at first when they only have time to review a limited number this cycle?

Unit of analysis: one content page (content_id), scored using its trailing 90-day window of signals.

Decision this improves: which pages enter a reviewer's queue, and in what order.

Action someone takes: a human reviewer opens the top-ranked pages and decides whether to refresh, expand, protect, prune, or monitor — the model doesn't act, it prioritizes.

Cost of a wrong call: if a page is ranked highly but isn't actually worth reviewing, that's wasted reviewer time (low cost). If a genuinely declining, high-traffic page is ranked low and missed, that's a real cost — lost visibility caught too late. This asymmetry means catching high-impact declining pages matters more than perfect precision everywhere.

Why ML helps at all: the hand-written baseline already captures some signal (Precision@50 = 0.240), but a learned model found a meaningfully better ranking (0.740) from the same inputs — the relationship between staleness, position, and impressions isn't simply additive the way the hand rule assumes."""


"Question: Which pages should a content reviewer look at first when they only have time to review a limited number this cycle?\n\nUnit of analysis: one content page (content_id), scored using its trailing 90-day window of signals.\n\nDecision this improves: which pages enter a reviewer's queue, and in what order.\n\nAction someone takes: a human reviewer opens the top-ranked pages and decides whether to refresh, expand, protect, prune, or monitor — the model doesn't act, it prioritizes.\n\nCost of a wrong call: if a page is ranked highly but isn't actually worth reviewing, that's wasted reviewer time (low cost). If a genuinely declining, high-traffic page is ranked low and missed, that's a real cost — lost visibility caught too late. This asymmetry means catching high-impact declining pages matters more than perfect precision everywhere.\n\nWhy ML helps at all: the hand-written baseline already captures some signal (Precision@50 = 0.240), but a learned model found a meaningfully better

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

declining_rate = (df["trend_direction"] == "down").mean()
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).mean()
median_impressions_declining = df.loc[df["trend_direction"] == "down", "impressions_90d"].median()

print(f"Share of pages currently declining: {declining_rate:.3f}")
print(f"Share of pages that are both stale (180+ days) and visible (500+ impressions): {stale_visible:.3f}")
print(f"Median 90-day impressions among declining pages: {median_impressions_declining:.0f}")


Share of pages currently declining: 0.542
Share of pages that are both stale (180+ days) and visible (500+ impressions): 0.001
Median 90-day impressions among declining pages: 961


In [ ]:
print(df["days_since_last_update"].describe())
print(df["impressions_90d"].describe())

count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64


In [ ]:
stale_only = (df["days_since_last_update"] >= 180).mean()
visible_only = (df["impressions_90d"] >= 500).mean()
print(f"Stale alone (180+ days): {stale_only:.3f}")
print(f"Visible alone (500+ impressions): {visible_only:.3f}")

Stale alone (180+ days): 0.006
Visible alone (500+ impressions): 0.558


In [ ]:
"""54.2% of pages in the starter dataset are currently declining (trend_direction == "down") — over half the inventory, not a rare edge case. Declining pages still carry real traffic: the median declining page has 961 impressions in 90 days, meaning most aren't dead pages, they're pages worth reviewing. Only 0.1% of pages are simultaneously stale (180+ days unchanged) and highly visible (500+ impressions) — but this is mostly because 180+ days of staleness is already an extreme value in this dataset (75th percentile is only 104 days), not because staleness and visibility are unrelated. This suggests a single hand-written rule threshold like "180 days stale" misses most of the real opportunity, which lines up with why the learned model (Precision@50 = 0.740) beat the simple hand-rule baseline (0.240) in Weeks 1–2 — the model can find graded, combined signal instead of relying on one hard cutoff."""

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""I can claim this ranking is decision-support, based on observed historical signals, evaluated with client-holdout validation. I cannot claim a refresh will cause recovery — that needs a causal experiment this data can't provide. I also can't claim to have found "the real reason" a page declined — only that certain signals are statistically associated with the current trend_direction == "down" proxy label. For the capstone, I plan to move toward a future-window label (prior 90 days → next 30 days decline) rather than this current-window proxy."""


'I can claim this ranking is decision-support, based on observed historical signals, evaluated with client-holdout validation. I cannot claim a refresh will cause recovery — that needs a causal experiment this data can\'t provide. I also can\'t claim to have found "the real reason" a page declined — only that certain signals are statistically associated with the current trend_direction == "down" proxy label. For the capstone, I plan to move toward a future-window label (prior 90 days → next 30 days decline) rather than this current-window proxy.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.